# Model I

Biblioteki

In [2]:
from datetime import datetime
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score
import time
from model import InstrumentCNN
from trening import get_device, train_model
from przygotowanie_danych import (INSTRUMENTS, create_datasets, create_loaders)

### Konfiguracja eksperymentu

In [ ]:
dataset = "1_1"
model_name = "CNN1"

epochs = 100
patience = 10
batch_size = 8
learning_rate = 0.001

dropout2 = 0.1
dropout3 = 0.3

threshold = 0.8

results_file = Path("wyniki/modele.csv")

Sprawdzenie, czy eksperyment był już wykonywany

In [5]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout 2"] == dropout2) &
        (results["Dropout 3"] == dropout3)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymuję program."
        )

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")


Nowa kombinacja. Kontynuuję wykonywanie kodu.


### Nadanie identyfikatora eksperymentu

In [7]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 1
Plik wag: 1.pth


### Przygotowanie danych

In [ ]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 7758
Validation: 1658
Test: 1664


### Inicjalizacja modelu

In [ ]:
device = get_device()
print(f"Urządzenie: {device}")

model = InstrumentCNN(dropout2=dropout2, dropout3=dropout3).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Urządzenie: mps


### Trening sieci

In [10]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

/opt/anaconda3/envs/cnn/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoka 1/100 | Train loss: 0.6485 | Val loss: 0.5732
Epoka 2/100 | Train loss: 0.5759 | Val loss: 0.5365
Epoka 3/100 | Train loss: 0.5511 | Val loss: 0.5174
Epoka 4/100 | Train loss: 0.5352 | Val loss: 0.4975
Epoka 5/100 | Train loss: 0.5210 | Val loss: 0.4892
Epoka 6/100 | Train loss: 0.5125 | Val loss: 0.4828
Epoka 7/100 | Train loss: 0.5053 | Val loss: 0.4831
Epoka 8/100 | Train loss: 0.4996 | Val loss: 0.4699
Epoka 9/100 | Train loss: 0.4939 | Val loss: 0.4659
Epoka 10/100 | Train loss: 0.4877 | Val loss: 0.4561
Epoka 11/100 | Train loss: 0.4826 | Val loss: 0.4504
Epoka 12/100 | Train loss: 0.4758 | Val loss: 0.4460
Epoka 13/100 | Train loss: 0.4699 | Val loss: 0.4413
Epoka 14/100 | Train loss: 0.4675 | Val loss: 0.4493
Epoka 15/100 | Train loss: 0.4600 | Val loss: 0.4417
Epoka 16/100 | Train loss: 0.4574 | Val loss: 0.4336
Epoka 17/100 | Train loss: 0.4511 | Val loss: 0.4242
Epoka 18/100 | Train loss: 0.4469 | Val loss: 0.4157
Epoka 19/100 | Train loss: 0.4424 | Val loss: 0.4136
Ep

### Zapis informacji o eksperymencie

In [12]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout 2": dropout2,
    "Dropout 3": dropout3,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")


Informacje o eksperymencie zapisano w: wyniki/modele.csv
Historia uczenia: history_1.csv
